# Interactive Results Exploration

This notebook demonstrates how to load and explore previously saved simulation results.

## Features:
- Load results from HDF5 files
- Interactive filtering by civilization status, Kardashev level
- Custom visualizations
- Compare multiple simulation runs

---

## 1. Setup

In [ ]:
from great_silence.notebook import (
    ResultsExplorer, 
    load_simulation, 
    reconstruct_simulation_from_hdf5,
    configure_notebook_display
)
import pandas as pd
import matplotlib.pyplot as plt

configure_notebook_display()

## 2. Load Simulation Results

### Option A: Load specific file

In [ ]:
# Load results from HDF5 file
data = load_simulation('output/my_simulation.h5')

# Display summary
print("Loaded simulation data:")
print(f"  Total stars: {len(data['galaxy']['positions'])}")
print(f"  Total civilizations: {len(data['civilizations'])}")
print(f"\nStatistics:")
for key, value in data['statistics'].items():
    print(f"  {key}: {value}")

### Reconstruct Simulation for Visualization

If you want to use visualization tools (like Interactive3DVisualizer), reconstruct the full simulation object:

In [ ]:
# Example: Create 3D visualization from loaded data
if 'sim' in locals():
    from great_silence.visualization import Interactive3DVisualizer
    
    viz = Interactive3DVisualizer(sim)
    
    # Create static figure
    fig = viz.create_static_figure(
        subsample_stars=5000,
        show_stars=True,
        show_active=True,
        show_extinct=True,
        show_deaths=True,
        show_hazards=True
    )
    
    fig.update_layout(title="Loaded Simulation: 3D Galaxy View")
    fig.show()
    
    print("\n💡 You can also use viz.create_animated_figure() for animations!")
else:
    print("⚠️ Reconstruct simulation first using the cell above")

### Visualize Loaded Simulation

Use the reconstructed simulation with any visualization tools:

In [ ]:
# Reconstruct full simulation object for visualization
sim = reconstruct_simulation_from_hdf5('output/my_simulation.h5')

print("✓ Simulation reconstructed from HDF5")
print(f"  Civilizations: {len(sim.civilizations)}")
print(f"  Galaxy stars: {len(sim.galaxy.positions)}")
print(f"  Hazard events: {len(sim.hazard_events) if hasattr(sim, 'hazard_events') else 0}")
print(f"  Snapshots: {len(sim.snapshots) if hasattr(sim, 'snapshots') else 0}")
print("\n✓ Ready for visualization with Interactive3DVisualizer!")

### Option B: Browse directory with widget

In [ ]:
# Create explorer with file browser
explorer = ResultsExplorer.from_directory('output/')
# Display interface
explorer.display();

## 3. Civilization Analysis

Convert civilization data to pandas DataFrame for analysis.

In [ ]:
# Create DataFrame from civilization data
civs_df = pd.DataFrame(data['civilizations'])

print(f"Total civilizations: {len(civs_df)}")
print(f"\nDataFrame info:")
civs_df.info()

print(f"\nFirst 10 civilizations:")
civs_df.head(10)

## 4. Filter and Analyze

### Active vs Extinct Civilizations

In [ ]:
active_civs = civs_df[civs_df['is_active'] == True]
extinct_civs = civs_df[civs_df['is_active'] == False]

print(f"Active civilizations: {len(active_civs)}")
print(f"Extinct civilizations: {len(extinct_civs)}")

# Survival rate
survival_rate = len(active_civs) / len(civs_df) * 100
print(f"\nSurvival rate: {survival_rate:.1f}%")

### Kardashev Level Distribution

In [ ]:
# Plot Kardashev level histogram
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.hist(active_civs['kardashev_level'], bins=20, alpha=0.7, label='Active', color='green')
plt.hist(extinct_civs['kardashev_level'], bins=20, alpha=0.7, label='Extinct', color='red')
plt.xlabel('Kardashev Level')
plt.ylabel('Count')
plt.title('Kardashev Level Distribution')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.scatter(civs_df['emergence_time_gyr'], civs_df['kardashev_level'], 
           c=civs_df['is_active'], cmap='RdYlGn', alpha=0.6)
plt.xlabel('Emergence Time (Gyr)')
plt.ylabel('Kardashev Level')
plt.title('Kardashev Level vs Emergence Time')
plt.colorbar(label='Active (1) / Extinct (0)')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Death Cause Analysis

In [ ]:
# Count death causes
death_causes = extinct_civs['death_cause'].value_counts()

print("Extinction causes:")
print(death_causes)

# Plot pie chart
plt.figure(figsize=(10, 7))
death_causes.plot.pie(autopct='%1.1f%%', startangle=90)
plt.title('Civilization Extinction Causes')
plt.ylabel('')
plt.show()

## 5. Spatial Distribution

Analyze where civilizations emerge in the galaxy.

In [ ]:
# Get civilization positions
import numpy as np

galaxy_positions = data['galaxy']['positions']
civ_indices = civs_df['parent_star_idx'].values
civ_positions = galaxy_positions[civ_indices]

# Calculate galactic radii
civ_radii = np.sqrt(civ_positions[:, 0]**2 + civ_positions[:, 1]**2)

# Plot radial distribution
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.hist(civ_radii, bins=30, alpha=0.7, edgecolor='black')
plt.xlabel('Galactic Radius (kpc)')
plt.ylabel('Number of Civilizations')
plt.title('Civilization Radial Distribution')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
active_positions = galaxy_positions[active_civs['parent_star_idx'].values]
extinct_positions = galaxy_positions[extinct_civs['parent_star_idx'].values]

plt.scatter(active_positions[:, 0], active_positions[:, 1], 
           c='green', s=10, alpha=0.6, label='Active')
plt.scatter(extinct_positions[:, 0], extinct_positions[:, 1], 
           c='red', s=10, alpha=0.4, label='Extinct')
plt.xlabel('X (kpc)')
plt.ylabel('Y (kpc)')
plt.title('Civilization Spatial Distribution')
plt.legend()
plt.axis('equal')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Lifetime Analysis

In [ ]:
import numpy as np

# Calculate lifetimes for extinct civilizations
extinct_civs_with_time = extinct_civs[extinct_civs['extinction_time_gyr'].notna()].copy()
extinct_civs_with_time['lifetime_gyr'] = (
    extinct_civs_with_time['extinction_time_gyr'] - 
    extinct_civs_with_time['emergence_time_gyr']
)

# Statistics
mean_lifetime = extinct_civs_with_time['lifetime_gyr'].mean()
median_lifetime = extinct_civs_with_time['lifetime_gyr'].median()

print(f"Civilization Lifetime Statistics:")
print(f"  Mean: {mean_lifetime:.3f} Gyr ({mean_lifetime * 1000:.1f} Myr)")
print(f"  Median: {median_lifetime:.3f} Gyr ({median_lifetime * 1000:.1f} Myr)")
print(f"  Min: {extinct_civs_with_time['lifetime_gyr'].min():.3f} Gyr")
print(f"  Max: {extinct_civs_with_time['lifetime_gyr'].max():.3f} Gyr")

# Plot distribution
plt.figure(figsize=(10, 6))
plt.hist(extinct_civs_with_time['lifetime_gyr'] * 1000, bins=30, 
         alpha=0.7, edgecolor='black')
plt.axvline(mean_lifetime * 1000, color='red', linestyle='--', 
           label=f'Mean: {mean_lifetime * 1000:.1f} Myr')
plt.axvline(median_lifetime * 1000, color='orange', linestyle='--', 
           label=f'Median: {median_lifetime * 1000:.1f} Myr')
plt.xlabel('Civilization Lifetime (Myr)')
plt.ylabel('Count')
plt.title('Distribution of Civilization Lifetimes')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 7. Compare Multiple Runs

Load and compare results from different presets or parameters.

In [ ]:
# Example: Compare optimistic vs realistic scenarios
# (Requires running simulations with different presets first)

# data_optimistic = load_simulation('output/optimistic_run.h5')
# data_realistic = load_simulation('output/realistic_run.h5')

# comparison_df = pd.DataFrame({
#     'Scenario': ['Optimistic', 'Realistic'],
#     'Total Civs': [
#         data_optimistic['statistics']['total_civilizations'],
#         data_realistic['statistics']['total_civilizations']
#     ],
#     'Active Civs': [
#         data_optimistic['statistics']['active_civilizations'],
#         data_realistic['statistics']['active_civilizations']
#     ]
# })

# display(comparison_df)

print("Load multiple simulation files and compare their statistics here!")

---

## Next Steps

- **01_quickstart_production_workflow.ipynb**: Run new simulations with metallicity targeting
- **03_animation_generation.ipynb**: Create timeline animations
- **04_expansion_trajectories_spheres.ipynb**: Visualize probe expansion patterns

### Custom Analysis Ideas

- **Metallicity-based expansion**: Correlate stellar metallicity with colonization success
- **Sensor retargeting efficiency**: Analyze how often probes change course
- **Expansion velocity vs Kardashev level**: Study technology-dependent colonization rates
- Analyze hazard zone proximity
- Study colonization patterns (sparse vs dense)
- Compare early vs late Great Filter effects
- Examine Kardashev progression rates
- **Resource-rich vs habitable targeting**: Compare probe targeting strategies

### Analyzing Probe Expansion Data

```python
# Example: Analyze colonized system metallicities
if 'colonized_systems' in data:
    colonized_indices = data['colonized_systems']
    metallicities = data['galaxy']['metallicities'][colonized_indices]
    
    print(f"Colonized systems: {len(colonized_indices)}")
    print(f"Metallicity range: {metallicities.min():.2f} to {metallicities.max():.2f} [Fe/H]")
    print(f"Mean metallicity: {metallicities.mean():.2f} [Fe/H]")
    
    # Compare to galaxy-wide distribution
    all_metallicities = data['galaxy']['metallicities']
    print(f"\nGalaxy-wide mean: {all_metallicities.mean():.2f} [Fe/H]")
```

Happy exploring! 🔍📊